# This notebook analyzes the mice sent to me on Feb 27 2022

In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf
import matplotlib.pyplot as plt
sys.path.append('/home/austin/Utilities/Code/Tensorflow/')
from utils_misc_tf import limitGPU
limitGPU(2048)

In [ ]:
print(tf.__version__)

## Import NMF model

In [ ]:
sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic


## Import my preprocessing code

In [ ]:
from data_tools import load_data

## Load the training data

In [ ]:
fnm='../Aggression_25quan_train_MOuse.mat'
power_train,coherence_train,granger_train,labels_train = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

granger_train = np.exp(granger_train)
granger_train[granger_train>10] = 10
power_train = power_train*10
power_train[power_train>6] = 6


In [ ]:
train_name = np.genfromtxt('Train_name.txt',dtype='U15')
train_group = np.genfromtxt('Train_group.txt')
train_expDate = np.genfromtxt('Train_expDate.txt')
train_condition = np.genfromtxt('Train_condition.txt')
train_behavior = np.genfromtxt('Train_behavior.txt')

print(train_name.shape)
print(train_group.shape)
print(train_expDate.shape)
print(train_condition.shape)
print(train_behavior.shape)

#### Positive = condition == 4 and behavior==1
#### Negative = condition == 4,6,8 and behavior==2

In [ ]:
train_idx_pos = (train_condition==4)&(train_behavior==1)
train_idx_neg = (train_behavior==2)&((train_condition==4)|(train_condition==6)|(train_condition==8))
train_idx_tot = train_idx_pos|train_idx_neg

y_total_train = np.zeros(len(train_name))
y_total_train[train_idx_pos] = 1

In [ ]:
X_train_total = np.hstack((power_train,coherence_train,granger_train))
X_train = X_train_total[train_idx_tot]
y_train = y_total_train[train_idx_tot]

## Load the testing data

In [ ]:
fnm='../Aggression_25quan_test_MOuse.mat'
power_test,coherence_test,granger_test,labels_test = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

granger_test = np.exp(granger_test)
granger_test[granger_test>10] = 10
power_test = power_test*10
power_test[power_test>6] = 6

In [ ]:
test_name = np.genfromtxt('Test_name.txt',dtype='U15')
test_group = np.genfromtxt('Test_group.txt')
test_expDate = np.genfromtxt('Test_expDate.txt')
test_condition = np.genfromtxt('Test_condition.txt')
test_behavior = np.genfromtxt('Test_behavior.txt')

print(test_name.shape)
print(test_group.shape)
print(test_expDate.shape)
print(test_condition.shape)
print(test_behavior.shape)

In [ ]:
test_idx_pos = (test_condition==4)&(test_behavior==1)
test_idx_neg = (test_behavior==2)&((test_condition==4)|(test_condition==6)|(test_condition==8))
test_idx_tot = test_idx_pos|test_idx_neg

y_total_test = np.zeros(len(test_name))
y_total_test[test_idx_pos] = 1

In [ ]:
X_test_total = np.hstack((power_test,coherence_test,granger_test))
X_test = X_test_total[test_idx_tot]
y_test = y_total_test[test_idx_tot]

## Actually run the model

In [ ]:
nFact = 8
nIter = 2500
mu = 1
model = NMF_logistic(nFact,nIter=nIter,LR=1e-3,mu=mu,batchSize=100)


In [ ]:
S_train_train = model.fit_transform(X_train,y_train)

In [ ]:
results_dict = {}

## Saving model parameters

In [ ]:
S_train = model.transform(X_train)
S_test = model.transform(X_test)

results_dict['S_train'] = S_train
results_dict['S_test'] = S_test
results_dict['W_'] = model.components_
results_dict['A_'] = model.A_enc
results_dict['B_'] = model.B_enc
results_dict['phi'] = model.Phi


In [ ]:
np.amin(S_train)

In [ ]:
print('Predictive coefficient is: ',model.Phi)

In [ ]:
Y_pred = np.squeeze(S_test[:,0]*model.Phi)

## Total ROC

In [ ]:
print(roc_auc_score(y_test,Y_pred))
results_dict['OverallROC'] = roc_auc_score(y_test,Y_pred)

In [ ]:
mouse_test = np.unique(test_name)
mouse_test_sub = test_name[test_idx_tot]
print(mouse_test)

In [ ]:
roc_test = np.zeros(len(mouse_test))
for i in range(len(mouse_test)):
    roc_test[i] = roc_auc_score(y_test[mouse_test_sub==mouse_test[i]],Y_pred[mouse_test_sub==mouse_test[i]])
    print(mouse_test[i],roc_test[i])

In [ ]:
results_dict['ROCs_test'] = roc_test
print(np.mean(roc_test))

### Test set ROCS

In [ ]:
roc_test_other = np.zeros((len(mouse_test),7))
for j in range(7):
    for i in range(len(mouse_test)):
        roc_test_other[i,j] = roc_auc_score(y_test[mouse_test_sub==mouse_test[i]],
                                          S_test[mouse_test_sub==mouse_test[i],j+1])


In [ ]:
np.mean(roc_test_other,axis=0)

In [ ]:
for i in range(len(mouse_test)):
    print(mouse_test[i],roc_test_other[i,4])

In [ ]:
np.mean(roc_test_other[:,4])

### Training set ROCS

In [ ]:
mouse_train = np.unique(train_name)
mouse_train_sub = train_name[train_idx_tot]
print(mouse_train)

roc_train_other = np.zeros((len(mouse_train),8))
smean = np.zeros((len(mouse_train),8))
for j in range(8):
    for i in range(len(mouse_train)):
        smean[i,j] = np.mean(S_train[mouse_train_sub==mouse_train[i],j])
        try:
            roc_train_other[i,j] = roc_auc_score(y_train[mouse_train_sub==mouse_train[i]],
                                          S_train[mouse_train_sub==mouse_train[i],j])
        except:
            roc_train_other[i,j] = -0.01

In [ ]:
for i in range(len(mouse_train)):
    print(mouse_train[i],roc_train_other[i,4])

In [ ]:
np.mean(roc_train_other[roc_train_other[:,4]>0,4])

In [ ]:
plt.plot(model.losses_sup)

In [ ]:
plt.plot(model.losses_recon)

In [ ]:
results_dict['roc_test_other'] = roc_test_other
pickle.dump(results_dict,open('Results_furtherSubset_allAUCs_earlyStopping.txt','wb'))